In [2]:
import os
import mlflow

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:3900"
os.environ["AWS_ACCESS_KEY_ID"]      = "mlflow-access-key"
os.environ["AWS_SECRET_ACCESS_KEY"]  = "mlflow-secret-key"
os.environ["AWS_DEFAULT_REGION"]     = "garage"

mlflow.set_tracking_uri("http://localhost:5000/")

In [3]:
mlflow.set_experiment("Exp 3 - TfIdf Trigram max_features")

2026/04/25 17:55:42 INFO mlflow.tracking.fluent: Experiment with name 'Exp 3 - TfIdf Trigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-artifacts/3', creation_time=1777121742847, experiment_id='3', last_update_time=1777121742847, lifecycle_stage='active', name='Exp 3 - TfIdf Trigram max_features', tags={}, trace_location=None, workspace='default'>

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [6]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [7]:
# Step 1: Function to run the experiment
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1, 3)  # Trigram setting

    # Step 2: Vectorization using TF-IDF with varying max_features
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"TFIDF_Trigrams_max_features_{max_features}")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, max_features={max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_{max_features}")

# Step 6: Test various max_features values
max_features_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

for max_features in max_features_values:
    run_experiment_tfidf_max_features(max_features)

2026/04/25 17:56:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 17:56:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_1000 at: http://localhost:5000/#/experiments/3/runs/fbc8eb104a154bfeb67a02e3f56669bd
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/04/25 17:56:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 17:56:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_2000 at: http://localhost:5000/#/experiments/3/runs/5733da07b632455ea9e93401b29bad6e
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/04/25 17:56:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 17:56:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run TFIDF_Trigrams_max_features_8000 at: http://localhost:5000/#/experiments/3/runs/624762d396d44528b7f469cc48b44429
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/04/25 17:58:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 17:58:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_9000 at: http://localhost:5000/#/experiments/3/runs/72b23d0cbfa64cb9ab778a80e8027b9a
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/04/25 17:58:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 17:58:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_10000 at: http://localhost:5000/#/experiments/3/runs/744260183bb44727bd4159e59e429412
🧪 View experiment at: http://localhost:5000/#/experiments/3
